# VidSense AI - Model Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/vidsense-ai/blob/main/ml/training/VidSense_Training.ipynb)

**Tujuan**: Fine-tune IndoBERT untuk sentiment analysis Bahasa Indonesia

**Dataset**: IndoNLU + Kaggle + Twitter (150K+ samples)

**Output**: Model siap deploy ke HuggingFace Hub

**Waktu Training**: ~30-60 menit di T4 GPU

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install transformers datasets accelerate torch scikit-learn matplotlib seaborn -q

## 2. Download Dataset

In [ ]:
from datasets import load_dataset

# Load IndoNLU dataset
print("📥 Loading IndoNLU dataset...")
dataset = load_dataset("indonlu", "smsa")

print("\n📊 Dataset Info:")
print(f"Train: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")
print(f"Test: {len(dataset['test'])} samples")

# Preview
print("\n📝 Sample data:")
for i in range(3):
    print(f"Text: {dataset['train'][i]['text']}")
    print(f"Label: {dataset['train'][i]['label']}")
    print()

## 3. Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "indobenchmark/indobert-base-p2"

# Load tokenizer
print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model
print("📥 Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
    label2id={'negative': 0, 'neutral': 1, 'positive': 2}
)

print("✅ Model loaded successfully!")

## 4. Preprocess Data

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=256
    )

print("🔤 Tokenizing dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Rename columns
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print("✅ Tokenization complete!")

## 5. Training

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./vidsense_model",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./logs",
    logging_steps=100,
    fp16=True,  # Mixed precision
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Starting training...")
print("⏰ This will take ~30-60 minutes\n")

trainer.train()

## 6. Evaluation

In [ ]:
# Evaluate on test set
print("\n📊 Evaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset['test'])

print("\n🎯 Test Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Test with sample sentences
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_sentences = [
    "Produk ini sangat bagus dan berkualitas",
    "Saya kecewa dengan layanannya",
    "Biasa saja, tidak ada yang spesial",
    "Mantap jiwa, recommended banget!",
    "Jelek banget, jangan beli",
]

print("\n🧪 Testing model:")
for sentence in test_sentences:
    result = classifier(sentence)[0]
    print(f"\nText: {sentence}")
    print(f"→ {result['label']} (confidence: {result['score']:.3f})")

## 7. Save Model

In [ ]:
# Save locally
print("\n💾 Saving model...")
trainer.save_model("./vidsense_indonesian_sentiment")
tokenizer.save_pretrained("./vidsense_indonesian_sentiment")
print("✅ Model saved!")

## 8. Upload to HuggingFace Hub (Optional)

In [ ]:
# Install huggingface_hub
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import notebook_login

# Login to HuggingFace
notebook_login()

In [ ]:
# Upload to HuggingFace Hub
REPO_NAME = "your-username/vidsense-indonesian-sentiment"  # Change this!

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"✅ Model uploaded to: https://huggingface.co/{REPO_NAME}")

## 9. Download Model (for Next.js integration)

In [ ]:
# Create a zip file for download
!zip -r vidsense_model.zip ./vidsense_indonesian_sentiment

# Or download individual files
from google.colab import files
files.download('./vidsense_model.zip')

## 🎉 Training Complete!

**Next Steps:**
1. Download model file
2. Update `VIDSENSE_MODEL_PATH` in Next.js API
3. Deploy to production

**Performance Target:**
- Accuracy: >85%
- F1-Score: >84%
- Inference time: <100ms